In [1]:
import yfinance as yf
import numpy as np
import pandas as pd
import os

In [2]:
data = pd.read_csv('../Data/BNBUSDT-EOHSummary-2023-10-23.csv')
data.columns = data.columns.str.strip()
data.tail()

,date,hour,symbol,underlying,type,strike,open,high,low,close,...,best_buy_iv,best_sell_iv,mark_price,mark_iv,delta,gamma,vega,theta,openinterest_contracts,openinterest_usdt
475,2023-10-23,17,BNB-231027-215-C,BNBUSDT,C,231027-215,5.7,7.8,4.9,6.0,...,NaN,0.371933,6.8,0.400000,0.745673,0.036688,0.070069,-0.391080,360.38,79494.602584
476,2023-10-23,4,BNB-231027-190-P,BNBUSDT,P,231027-190,0.1,0.2,0.1,0.2,...,NaN,0.775324,0.1,0.734815,-0.021891,0.003018,0.012320,-0.109734,1107.28,245647.571992
477,2023-10-23,2,BNB-231027-190-C,BNBUSDT,C,231027-190,23.2,23.2,23.2,23.2,...,NaN,1.712894,31.2,1.056447,0.912665,0.006357,0.037547,-0.471285,10.02,2205.351687
478,2023-10-23,15,BNB-231027-205-C,BNBUSDT,C,231027-205,11.5,11.9,11.5,11.9,...,NaN,0.852532,15.2,0.626266,0.865191,0.015759,0.047671,-0.407111,7.49,1642.568654
479,2023-10-23,21,BNB-231027-230-C,BNBUSDT,C,231027-230,1.2,1.2,0.7,1.2,...,NaN,0.481087,1.6,0.440543,0.277420,0.035087,0.072652,-0.468381,1374.85,308193.545582


In [3]:
data = data.sort_values(["date", "hour"])

In [4]:
data.columns

Index(['date', 'hour', 'symbol', 'underlying', 'type', 'strike', 'open',
       'high', 'low', 'close', 'volume_contracts', 'volume_usdt',
       'best_bid_price', 'best_ask_price', 'best_bid_qty', 'best_ask_qty',
       'best_buy_iv', 'best_sell_iv', 'mark_price', 'mark_iv', 'delta',
       'gamma', 'vega', 'theta', 'openinterest_contracts',
       'openinterest_usdt'],
      dtype='object')

In [5]:
data["strike_price"] = data["strike"].str.split("-").str[1].astype(float)
# We only want call options
data = data[data['type']=='C']

In [6]:
data = data.iloc[::-1]
data.tail()

,date,hour,symbol,underlying,type,strike,open,high,low,close,...,best_sell_iv,mark_price,mark_iv,delta,gamma,vega,theta,openinterest_contracts,openinterest_usdt,strike_price
231,2023-10-23,0,BNB-231027-200-C,BNBUSDT,C,231027-200,16.4,17.7,16.4,17.7,...,0.855026,18.9,0.627513,0.906820,0.011217,0.039444,-0.288364,21.00,4585.494939,200.0
203,2023-10-23,0,BNB-231027-190-C,BNBUSDT,C,231027-190,23.2,23.2,23.2,23.2,...,1.545362,29.2,0.972681,0.914596,0.006782,0.036967,-0.418922,10.02,2187.936157,190.0
187,2023-10-23,0,BNB-231027-210-C,BNBUSDT,C,231027-210,6.4,6.4,6.4,6.4,...,0.423353,9.3,0.411676,0.813729,0.027507,0.063457,-0.304356,13.00,2838.639724,210.0
163,2023-10-23,0,BNB-231027-230-C,BNBUSDT,C,231027-230,0.9,1.0,0.5,0.7,...,0.502102,0.9,0.476298,0.162523,0.021801,0.058188,-0.322893,1246.99,272288.873050,230.0
148,2023-10-23,0,BNB-231027-220-C,BNBUSDT,C,231027-220,1.7,2.7,1.7,2.7,...,0.367072,3.0,0.400000,0.437989,0.041622,0.093296,-0.434775,400.48,87447.572057,220.0


In [7]:
S_data = pd.read_csv('../Data/BNBUSDT-1h-2023-10-23.csv', header=None)
S_data

,0,1,2,3,4,5,6,7,8,9,10,11
0,1698019200000,217.8,218.6,216.9,218.2,19734.256,1698022799999,4.298515e+06,7316,11125.938,2.423694e+06,0
1,1698022800000,218.3,219.9,218.1,218.9,43378.296,1698026399999,9.505224e+06,12711,23545.765,5.159363e+06,0
2,1698026400000,218.9,220.6,218.8,220.2,35051.992,1698029999999,7.705247e+06,12666,22289.151,4.899772e+06,0
3,1698030000000,220.2,222.8,219.6,222.2,69389.598,1698033599999,1.537461e+07,19221,39759.005,8.808892e+06,0
4,1698033600000,222.2,222.7,221.3,221.7,27256.135,1698037199999,6.045863e+06,9327,12966.447,2.876381e+06,0
5,1698037200000,221.8,222.8,221.0,221.8,33569.052,1698040799999,7.461921e+06,9080,19305.147,4.292020e+06,0
6,1698040800000,221.9,222.7,220.1,220.2,25364.961,1698044399999,5.619037e+06,8744,9870.812,2.187219e+06,0
7,1698044400000,220.2,220.6,218.7,219.5,25417.886,1698047999999,5.591198e+06,9611,12566.885,2.764801e+06,0
8,1698048000000,219.5,220.9,219.0,220.6,16480.245,1698051599999,3.624109e+06,7051,9140.991,2.010350e+06,0
9,1698051600000,220.7,220.7,218.2,219.4,24998.096,1698055199999,5.483193e+06,10517,12613.531,2.766136e+06,0


In [8]:
S_data["datetime"] = pd.to_datetime(S_data.iloc[:,0], unit="ms", utc=True)

S_data["date"] = S_data["datetime"].dt.strftime("%Y-%m-%d")
S_data["hour"] = S_data["datetime"].dt.hour
S_data['Underlying Value'] = S_data[4]

In [9]:
S_data

,0,1,2,3,4,5,6,7,8,9,10,11,datetime,date,hour,Underlying Value
0,1698019200000,217.8,218.6,216.9,218.2,19734.256,1698022799999,4.298515e+06,7316,11125.938,2.423694e+06,0,2023-10-23 00:00:00+00:00,2023-10-23,0,218.2
1,1698022800000,218.3,219.9,218.1,218.9,43378.296,1698026399999,9.505224e+06,12711,23545.765,5.159363e+06,0,2023-10-23 01:00:00+00:00,2023-10-23,1,218.9
2,1698026400000,218.9,220.6,218.8,220.2,35051.992,1698029999999,7.705247e+06,12666,22289.151,4.899772e+06,0,2023-10-23 02:00:00+00:00,2023-10-23,2,220.2
3,1698030000000,220.2,222.8,219.6,222.2,69389.598,1698033599999,1.537461e+07,19221,39759.005,8.808892e+06,0,2023-10-23 03:00:00+00:00,2023-10-23,3,222.2
4,1698033600000,222.2,222.7,221.3,221.7,27256.135,1698037199999,6.045863e+06,9327,12966.447,2.876381e+06,0,2023-10-23 04:00:00+00:00,2023-10-23,4,221.7
5,1698037200000,221.8,222.8,221.0,221.8,33569.052,1698040799999,7.461921e+06,9080,19305.147,4.292020e+06,0,2023-10-23 05:00:00+00:00,2023-10-23,5,221.8
6,1698040800000,221.9,222.7,220.1,220.2,25364.961,1698044399999,5.619037e+06,8744,9870.812,2.187219e+06,0,2023-10-23 06:00:00+00:00,2023-10-23,6,220.2
7,1698044400000,220.2,220.6,218.7,219.5,25417.886,1698047999999,5.591198e+06,9611,12566.885,2.764801e+06,0,2023-10-23 07:00:00+00:00,2023-10-23,7,219.5
8,1698048000000,219.5,220.9,219.0,220.6,16480.245,1698051599999,3.624109e+06,7051,9140.991,2.010350e+06,0,2023-10-23 08:00:00+00:00,2023-10-23,8,220.6
9,1698051600000,220.7,220.7,218.2,219.4,24998.096,1698055199999,5.483193e+06,10517,12613.531,2.766136e+06,0,2023-10-23 09:00:00+00:00,2023-10-23,9,219.4


In [10]:
S = S_data[['date', 'hour', 'Underlying Value']]
S.to_csv("../S_path.csv", index=False)

In [11]:
data.head()

,date,hour,symbol,underlying,type,strike,open,high,low,close,...,best_sell_iv,mark_price,mark_iv,delta,gamma,vega,theta,openinterest_contracts,openinterest_usdt,strike_price
462,2023-10-23,23,BNB-231027-210-C,BNBUSDT,C,231027-210,6.4,16.6,6.4,16.6,...,0.868845,19.0,0.634422,0.923024,0.010421,0.031534,-0.300089,26.35,6016.151794,210.0
319,2023-10-23,23,BNB-231027-190-C,BNBUSDT,C,231027-190,23.2,23.2,23.2,23.2,...,2.181282,39.0,1.200000,0.952323,0.003788,0.021685,-0.390321,10.02,2287.735900,190.0
276,2023-10-23,23,BNB-231027-180-C,BNBUSDT,C,231027-180,33.4,48.0,33.4,48.0,...,2.964058,48.6,1.200000,0.983794,0.001544,0.008839,-0.159094,2.17,495.447795,180.0
211,2023-10-23,23,BNB-231027-230-C,BNBUSDT,C,231027-230,1.2,4.4,0.7,4.4,...,0.675311,4.0,0.537656,0.460837,0.033811,0.086709,-0.699292,1374.85,313901.567108,230.0
179,2023-10-23,23,BNB-231027-215-C,BNBUSDT,C,231027-215,5.7,13.6,4.9,13.6,...,0.972443,14.9,0.686221,0.832486,0.016726,0.054746,-0.563520,360.23,82246.617100,215.0


In [12]:
data["expiry"] = pd.to_datetime(
    "20" + data["symbol"].str.extract(r"-(\d{6})-")[0],
    format="%Y%m%d"
).dt.strftime("%Y-%m-%d")

In [13]:
C = data[['date', 'hour', 'expiry', 'strike_price', 'close']].copy()
C.head()

,date,hour,expiry,strike_price,close
462,2023-10-23,23,2023-10-27,210.0,16.6
319,2023-10-23,23,2023-10-27,190.0,23.2
276,2023-10-23,23,2023-10-27,180.0,48.0
211,2023-10-23,23,2023-10-27,230.0,4.4
179,2023-10-23,23,2023-10-27,215.0,13.6


In [14]:
C['close'] = C['close'].replace('-', np.nan)
C = C[C["close"] != "-"].copy()
C.to_csv("../C_grid.csv", index=False)

In [15]:
!pip install pandas openpyxl

In [16]:
r_Data = pd.read_excel('../Data/sofr_rates.xlsx')
r_Data["Date"] = pd.to_datetime(r_Data["Effective Date"], format="%m/%d/%Y")
r_Data = r_Data.sort_values("Date")
r_Data

/opt/anaconda3/envs/cwq/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Effective Date,Rate Type,Rate (%),1st Percentile (%),25th Percentile (%),75th Percentile (%),99th Percentile (%),Volume ($Billions),Target Rate From (%),Target Rate To (%),Intra Day - Low (%),Intra Day - High (%),Standard Deviation (%),30-Day Average SOFR,90-Day Average SOFR,180-Day Average SOFR,SOFR Index,Revision Indicator (Y/N),Footnote ID,Date
248,01/03/2023,SOFR,4.31,4.22,4.28,4.35,4.44,1257,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023-01-03
247,01/04/2023,SOFR,4.30,4.22,4.28,4.35,4.44,1136,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023-01-04
246,01/05/2023,SOFR,4.31,4.23,4.29,4.35,4.45,1134,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023-01-05
245,01/06/2023,SOFR,4.31,4.22,4.29,4.35,4.45,1118,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023-01-06
244,01/09/2023,SOFR,4.31,4.23,4.29,4.34,4.45,1127,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023-01-09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4,12/22/2023,SOFR,5.32,5.27,5.31,5.38,5.42,1667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023-12-22
3,12/26/2023,SOFR,5.35,5.28,5.31,5.45,5.48,1644,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023-12-26
2,12/27/2023,SOFR,5.39,5.29,5.33,5.49,5.55,1722,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023-12-27
1,12/28/2023,SOFR,5.40,5.30,5.35,5.55,5.64,1832,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023-12-28


In [17]:
full_dates = pd.date_range(
    start=r_Data["Date"].min(),
    end=r_Data["Date"].max(),
    freq="D"
)

# Forward fill
r_Data = (
    r_Data.set_index("Date")
      .reindex(full_dates)
      .ffill()
      .reset_index()
)

r_Data = r_Data.rename(columns={"index": "Date"})

r_Data["Date"] = r_Data["Date"].dt.strftime("%Y-%m-%d")

In [18]:
r_Data.head(10)

,Date,Effective Date,Rate Type,Rate (%),1st Percentile (%),25th Percentile (%),75th Percentile (%),99th Percentile (%),Volume ($Billions),Target Rate From (%),Target Rate To (%),Intra Day - Low (%),Intra Day - High (%),Standard Deviation (%),30-Day Average SOFR,90-Day Average SOFR,180-Day Average SOFR,SOFR Index,Revision Indicator (Y/N),Footnote ID
0,2023-01-03,01/03/2023,SOFR,4.31,4.22,4.28,4.35,4.44,1257.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-01-04,01/04/2023,SOFR,4.30,4.22,4.28,4.35,4.44,1136.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-01-05,01/05/2023,SOFR,4.31,4.23,4.29,4.35,4.45,1134.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023-01-06,01/06/2023,SOFR,4.31,4.22,4.29,4.35,4.45,1118.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023-01-07,01/06/2023,SOFR,4.31,4.22,4.29,4.35,4.45,1118.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2023-01-08,01/06/2023,SOFR,4.31,4.22,4.29,4.35,4.45,1118.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2023-01-09,01/09/2023,SOFR,4.31,4.23,4.29,4.34,4.45,1127.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2023-01-10,01/10/2023,SOFR,4.31,4.23,4.29,4.34,4.41,1124.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2023-01-11,01/11/2023,SOFR,4.30,4.23,4.28,4.34,4.41,1142.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2023-01-12,01/12/2023,SOFR,4.30,4.22,4.29,4.34,4.41,1108.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
r_Data.columns

Index(['Date', 'Effective Date', 'Rate Type', 'Rate (%)', '1st Percentile (%)',
       '25th Percentile (%)', '75th Percentile (%)', '99th Percentile (%)',
       'Volume ($Billions)', 'Target Rate From (%)', 'Target Rate To (%)',
       'Intra Day - Low (%)', 'Intra Day - High (%)', 'Standard Deviation (%)',
       '30-Day Average SOFR', '90-Day Average SOFR', '180-Day Average SOFR',
       'SOFR Index', 'Revision Indicator (Y/N)', 'Footnote ID'],
      dtype='object')

In [20]:
r = r_Data[['Date', 'Rate (%)']]
r.head()

,Date,Rate (%)
0,2023-01-03,4.31
1,2023-01-04,4.30
2,2023-01-05,4.31
3,2023-01-06,4.31
4,2023-01-07,4.31


In [21]:
r.to_csv('../r_path.csv', index=False)